<a href="https://colab.research.google.com/github/weso500/1B-Shared/blob/main/ZeroOverlap_Compareruns_EnsemblefPCAGMM_OTHER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pyreadr

In [ ]:
pip install scikit-fda

In [ ]:
import pyreadr
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from skfda import FDataGrid
from skfda.preprocessing.dim_reduction import FPCA
from skfda.misc.covariances import Exponential, Gaussian
from skfda.misc.metrics import l2_distance, l2_norm
from sklearn import metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from matplotlib import pyplot
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
import math
from sklearn.metrics import classification_report

In [ ]:
def round_down_to_decimals(number, num_decimals):
    multiplier = 10 ** num_decimals
    return math.floor(number * multiplier) / multiplier

In [ ]:
import pywt
import numpy as np
import pandas as pd
import pyreadr
from sklearn.preprocessing import StandardScaler


fault_numbers_to_select = [3,9,15]
# Load data
df_FaultFree = pyreadr.read_r(r'/content/drive/MyDrive/FDA Journal/TEP_FaultFree_Training.RData')['fault_free_training']
df_Faulty = pyreadr.read_r(r'/content/drive/MyDrive/FDA Journal/TEP_Faulty_Training.RData')['faulty_training']

df_train_good = df_FaultFree.iloc[:, 3:]
df_test_bad_raw = df_Faulty.loc[
    (df_Faulty['faultNumber'].isin(fault_numbers_to_select)) &
    (df_Faulty['sample'] > 20.0),
    df_Faulty.columns[3:]
]
# Create sliding windows for good data
stride = 200 # Modified stride to be equal to window_size
window_size = 200
windows_good = np.lib.stride_tricks.sliding_window_view(df_train_good, window_shape=(window_size, 1))
strided_windows_good = windows_good[::stride]
df_sliding_window_good = pd.DataFrame(strided_windows_good.reshape(strided_windows_good.shape[0], -1))

# Create sliding windows for bad data
windows_bad = np.lib.stride_tricks.sliding_window_view(df_test_bad_raw, window_shape=(window_size, 1))
strided_windows_bad = windows_bad[::stride]
df_sliding_window_bad = pd.DataFrame(strided_windows_bad.reshape(strided_windows_bad.shape[0], -1))

# Normalize the wavelet features
scaler = StandardScaler()
df_scaled_good = scaler.fit_transform(df_sliding_window_good)

# Use the same scaler fitted on good data to transform the bad data
df_scaled_bad = scaler.transform(df_sliding_window_bad)


print("Shape of scaled good data:", df_scaled_good.shape)
print("Shape of scaled bad data:", df_scaled_bad.shape)

Shape of scaled good data: (1250, 10400)
Shape of scaled bad data: (3600, 10400)


In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np
num_train_samples_good = 750
# Use np.random.choice to select indices and then index the array
good_indices = np.random.choice(df_scaled_good.shape[0], size=num_train_samples_good, replace=False)
X_train_scaled = df_scaled_good[good_indices] # Use selected good samples for training

# Use remaining good samples for testing
all_good_indices = np.arange(df_scaled_good.shape[0])
remaining_good_indices = np.setdiff1d(all_good_indices, good_indices)
X_test_good_scaled = df_scaled_good[remaining_good_indices]


# Randomly sample 10 samples from df_scaled_bad for testing
num_test_samples_bad = 10
# Use np.random.choice to select indices and then index the array
bad_indices = np.random.choice(df_scaled_bad.shape[0], size=num_test_samples_bad, replace=False)
X_test_bad_scaled = df_scaled_bad[bad_indices]


X_test_scaled = np.concatenate((X_test_good_scaled, X_test_bad_scaled), axis=0)

# Create labels for the test set
test_good_labels = [0] * X_test_good_scaled.shape[0]
test_bad_labels = [1] * X_test_bad_scaled.shape[0]
all_test_labels = test_good_labels + test_bad_labels


# Create a list to store the fitted FPCA models for each window
fpca_models = []
# List to store the number of components used for each window (will be 5 for all)
# n_components_list = [] # Not needed as n_components is fixed at 5
df_scaled_weights_list_train = [] # To store weights for training data
df_scaled_weights_list_test = [] # To store weights for test data


window_size = 200 # Define window size for clarity

# Iterate through the columns in steps of window_size
for i in range(0, X_train_scaled.shape[1], window_size):
    # Select the columns for the current window for training and testing
    window_train_data = X_train_scaled[:, i:i + window_size] # Use numpy slicing for array
    window_test_data = X_test_scaled[:, i:i + window_size]

    # Convert the window data to FDataGrid for FPCA
    window_train_fdata = FDataGrid(window_train_data)
    window_test_fdata = FDataGrid(window_test_data)

    test = FPCA(n_components=199)
    test.fit(window_train_fdata)
    explained_variance_ratio = test.explained_variance_ratio_

    # Compute cumulative explained variance
    cumulative_variance = np.cumsum(explained_variance_ratio)

    # Find the number of components needed to reach at least 90% variance
    q = np.argmax(cumulative_variance >= 0.5) + 1
    # Fit FPCA to the current window's training data with 5 components
    fpca_window = FPCA(n_components=q) # Set n_components to 5
    fpca_window.fit(window_train_fdata)
    fpca_models.append(fpca_window)
    # n_components_list.append(fpca_window.n_components_) # This caused the error, removed


    # Transform both training and test data using the fitted FPCA model
    window_weights_train = fpca_window.transform(window_train_fdata)
    window_weights_test = fpca_window.transform(window_test_fdata)

    df_scaled_weights_list_train.append(window_weights_train)
    df_scaled_weights_list_test.append(window_weights_test)


# Concatenate the weights from all windows for training and testing
X_train_transformed = np.concatenate(df_scaled_weights_list_train, axis=1)
X_test_transformed = np.concatenate(df_scaled_weights_list_test, axis=1)

print("Shape of X_train_transformed:", X_train_transformed.shape)
print("Shape of X_test_transformed:", X_test_transformed.shape)
lowest_bic = np.inf
bic = []
n_components_range = range(1, 2)
best_gmm = None

for n in n_components_range:
    gmm = GaussianMixture(n_components=n, random_state=42)
    gmm.fit(X_train_transformed)
    bic_score = gmm.bic(X_train_transformed)
    bic.append(bic_score)
    print(n)
    if bic_score < lowest_bic:
        lowest_bic = bic_score
        best_gmm = gmm

print(f"Best number of components: {best_gmm.n_components}")

Shape of X_train_transformed: (750, 1432)
Shape of X_test_transformed: (510, 1432)
1
Best number of components: 1


In [ ]:
import numpy as np
import pandas as pd
import pyreadr
import pywt
from sklearn import metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from skfda import FDataGrid
from skfda.preprocessing.dim_reduction import FPCA # Using skfda.FPCA again
from sklearn import mixture

def round_down_to_decimals(number, decimals):
    factor = 10 ** decimals
    return np.floor(number * factor) / factor

# Reconstruction error is not needed for this approach
# def calculate_reconstruction_error(model, data):
#     # Ensure data is FDataGrid for FPCA
#     data_fdata = FDataGrid(data)
#     recon_fdata = model.inverse_transform(model.transform(data_fdata))
#     return np.linalg.norm(data_fdata.values - recon_fdata.values, axis=1)

auctotal = []
f1total = []
precisiontotal = []
recalltotal = []
accuracytotal = []

for j in range(100): # Keep this loop for potential multiple runs if needed later, currently runs once
    num_train_samples_good = 750
    # Use np.random.choice to select indices and then index the array
    good_indices = np.random.choice(df_scaled_good.shape[0], size=num_train_samples_good, replace=False)
    X_train_scaled = df_scaled_good[good_indices] # Use selected good samples for training

    # Use remaining good samples for testing
    all_good_indices = np.arange(df_scaled_good.shape[0])
    remaining_good_indices = np.setdiff1d(all_good_indices, good_indices)
    X_test_good_scaled = df_scaled_good[remaining_good_indices]


    # Randomly sample 10 samples from df_scaled_bad for testing
    num_test_samples_bad = 30
    # Use np.random.choice to select indices and then index the array
    bad_indices = np.random.choice(df_scaled_bad.shape[0], size=num_test_samples_bad, replace=False)
    X_test_bad_scaled = df_scaled_bad[bad_indices]


    X_test_scaled = np.concatenate((X_test_good_scaled, X_test_bad_scaled), axis=0)

    # Create labels for the test set
    test_good_labels = [0] * X_test_good_scaled.shape[0]
    test_bad_labels = [1] * X_test_bad_scaled.shape[0]
    all_test_labels = test_good_labels + test_bad_labels


    # Create a list to store the fitted FPCA models for each window
    fpca_models = []
    # List to store the number of components used for each window (will be 5 for all)
    # n_components_list = [] # Not needed as n_components is fixed at 5
    df_scaled_weights_list_train = [] # To store weights for training data
    df_scaled_weights_list_test = [] # To store weights for test data


    window_size = 200 # Define window size for clarity

    # Iterate through the columns in steps of window_size
    for i in range(0, X_train_scaled.shape[1], window_size):
        # Select the columns for the current window for training and testing
        window_train_data = X_train_scaled[:, i:i + window_size] # Use numpy slicing for array
        window_test_data = X_test_scaled[:, i:i + window_size]

        # Convert the window data to FDataGrid for FPCA
        window_train_fdata = FDataGrid(window_train_data)
        window_test_fdata = FDataGrid(window_test_data)

        test = FPCA(n_components=199)
        test.fit(window_train_fdata)
        explained_variance_ratio = test.explained_variance_ratio_

        # Compute cumulative explained variance
        cumulative_variance = np.cumsum(explained_variance_ratio)

        # Find the number of components needed to reach at least 90% variance
        q = np.argmax(cumulative_variance >= 0.25) + 1
        # Fit FPCA to the current window's training data with 5 components
        fpca_window = FPCA(n_components=q) # Set n_components to 5
        fpca_window.fit(window_train_fdata)
        fpca_models.append(fpca_window)
        # n_components_list.append(fpca_window.n_components_) # This caused the error, removed


        # Transform both training and test data using the fitted FPCA model
        window_weights_train = fpca_window.transform(window_train_fdata)
        window_weights_test = fpca_window.transform(window_test_fdata)

        df_scaled_weights_list_train.append(window_weights_train)
        df_scaled_weights_list_test.append(window_weights_test)


    # Concatenate the weights from all windows for training and testing
    X_train_transformed = np.concatenate(df_scaled_weights_list_train, axis=1)
    X_test_transformed = np.concatenate(df_scaled_weights_list_test, axis=1)



    # Train Local Outlier Factor (LOF) model on the transformed training data
    # Using novelty=True for semi-supervised anomaly detection
    clf = mixture.GaussianMixture()
    clf.fit(X_train_transformed)

    # Predict anomalies on the transformed test data
    # decision_function returns the outlier scores (lower is more normal)
    # We want higher scores for anomalies, so multiply by -1
    err_test_total = clf.score_samples(X_test_transformed) * -1


    # Determine optimal threshold using ROC curve
    fpr, tpr, thresholds = metrics.roc_curve(all_test_labels, err_test_total, pos_label=1)
    optimal_idx = np.argmax(tpr - fpr)
    loss_threshold = round_down_to_decimals(thresholds[optimal_idx], 13)
    print(j)

    # Predict anomalies based on the threshold
    Fulloutput = [1 if err > loss_threshold else 0 for err in err_test_total]

    # Evaluate performance
    cm = confusion_matrix(all_test_labels, Fulloutput)
    f1 = f1_score(all_test_labels, Fulloutput)
    auc = metrics.auc(fpr, tpr)
    precision_score = metrics.precision_score(all_test_labels, Fulloutput)
    recall_score = metrics.recall_score(all_test_labels, Fulloutput)
    accuracy = metrics.accuracy_score(all_test_labels, Fulloutput)
    auctotal.append(auc)
    f1total.append(f1)
    print(f"F1 Score: {f1:.3f}")
    print(f"AUC: {auc:.3f}")
    precisiontotal.append(precision_score)
    recalltotal.append(recall_score)
    accuracytotal.append(accuracy)

# Print average results if the outer loop runs more than once
if len(auctotal) > 1:
    print("\nAverage Results over 100 runs:")
    print(sum(auctotal)/100)
    print(sum(f1total)/100)
    print(sum(precisiontotal)/100)
    print(sum(recalltotal)/100)
    print(sum(accuracytotal)/100)
else:
    # Print results for the single run
    print("\nResults for the single run:")
    print(f"AUC: {auctotal[0]}")
    print(f"F1 Score: {f1total[0]}")
    print(f"Precision: {precisiontotal[0]}")
    print(f"Recall: {recalltotal[0]}")
    print(f"Accuracy: {accuracytotal[0]}")



0
F1 Score: 0.126
AUC: 0.573
1
F1 Score: 0.156
AUC: 0.611
2
F1 Score: 0.238
AUC: 0.501
3
F1 Score: 0.165
AUC: 0.562
4
F1 Score: 0.168
AUC: 0.599
5
F1 Score: 0.130
AUC: 0.595
6
F1 Score: 0.166
AUC: 0.654
7
F1 Score: 0.160
AUC: 0.605
8
F1 Score: 0.145
AUC: 0.514
9
F1 Score: 0.149
AUC: 0.645
10
F1 Score: 0.133
AUC: 0.590
11
F1 Score: 0.219
AUC: 0.612
12
F1 Score: 0.173
AUC: 0.566
13
F1 Score: 0.157
AUC: 0.582
14
F1 Score: 0.141
AUC: 0.618
15
F1 Score: 0.199
AUC: 0.648
16
F1 Score: 0.240
AUC: 0.563
17
F1 Score: 0.158
AUC: 0.639
18
F1 Score: 0.259
AUC: 0.651
19
F1 Score: 0.214
AUC: 0.620
20
F1 Score: 0.165
AUC: 0.571
21
F1 Score: 0.194
AUC: 0.650
22
F1 Score: 0.146
AUC: 0.593
23
F1 Score: 0.208
AUC: 0.710
24
F1 Score: 0.286
AUC: 0.533
25
F1 Score: 0.250
AUC: 0.596
26
F1 Score: 0.237
AUC: 0.624
27
F1 Score: 0.144
AUC: 0.642
28
F1 Score: 0.194
AUC: 0.621
29
F1 Score: 0.139
AUC: 0.578
30
F1 Score: 0.149
AUC: 0.576
31
F1 Score: 0.339
AUC: 0.654
32
F1 Score: 0.221
AUC: 0.632
33
F1 Score: 0.165
A

In [ ]:
import numpy as np
import pandas as pd
import pyreadr
import pywt
from sklearn import metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

def round_down_to_decimals(number, decimals):
    factor = 10 ** decimals
    return np.floor(number * factor) / factor

def calculate_reconstruction_error(model, data):
    recon = model.inverse_transform(model.transform(data))
    return np.linalg.norm(data - recon, axis=1)

# Assuming df_scaled_good and df_scaled_bad are defined and scaled as in the previous cells

# Define the number of good samples for training and the ratio for the test set
num_train_samples_good = 500
test_ratio_good_to_bad = 100

# Create labels for the scaled data
labels_good = np.zeros(df_scaled_good.shape[0])
labels_bad = np.ones(df_scaled_bad.shape[0])

# Split the good data into training and a pool for testing
X_train_scaled, X_good_test_pool, y_train, y_good_test_pool = train_test_split(
    df_scaled_good, labels_good, train_size=num_train_samples_good, random_state=42, stratify=labels_good
)

# Determine the number of good samples needed for the test set based on the ratio and available bad samples
num_bad_samples_for_test = df_scaled_bad.shape[0]
num_good_samples_for_test = num_bad_samples_for_test * test_ratio_good_to_bad

# Ensure we don't request more good test samples than available in the pool
if num_good_samples_for_test > X_good_test_pool.shape[0]:
    print(f"Warning: Requested {num_good_samples_for_test} good test samples (based on {test_ratio_good_to_bad}:1 ratio), but only {X_good_test_pool.shape[0]} good samples are available in the test pool. Using all available good samples from the test pool for the test set good samples.")
    num_good_samples_for_test = X_good_test_pool.shape[0]

# Randomly sample good test samples from the test pool
good_test_indices = np.random.choice(X_good_test_pool.shape[0], size=num_good_samples_for_test, replace=False)
X_test_good_scaled = X_good_test_pool[good_test_indices]
y_test_good = y_good_test_pool[good_test_indices]

# Randomly sample bad test samples (using all available bad samples as per calculation)
bad_test_indices = np.random.choice(df_scaled_bad.shape[0], size=num_bad_samples_for_test, replace=False)
X_test_bad_scaled = df_scaled_bad[bad_test_indices]
y_test_bad = labels_bad[bad_test_indices]

# Combine the selected good and bad test samples into the final test set
X_test_scaled = np.concatenate((X_test_good_scaled, X_test_bad_scaled), axis=0)
y_test = np.concatenate((y_test_good, y_test_bad), axis=0)

# Shuffle the test set to mix good and bad samples
shuffle_index = np.random.permutation(X_test_scaled.shape[0])
X_test_scaled = X_test_scaled[shuffle_index]
y_test = y_test[shuffle_index]

print("Shape of X_train_scaled:", X_train_scaled.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test_good_scaled (from pool):", X_test_good_scaled.shape)
print("Shape of X_test_bad_scaled (from df_scaled_bad):", X_test_bad_scaled.shape)
print("Shape of X_test_scaled (combined good and bad):", X_test_scaled.shape)
print("Shape of y_test (combined good and bad):", y_test.shape)

# Fit PCA model on the training data (good samples only)
pca_model = PCA(n_components=300)
pca_model.fit(X_train_scaled)

# Plot cumulative explained variance
plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(pca_model.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.title('Cumulative Explained Variance by PCA Components')
plt.grid(True)
plt.axvline(x=9, color='r', linestyle='--', label='9 Components') # Add vertical line at x=10
plt.legend() # Add legend to show the label for the vertical line
plt.show()


# Calculate reconstruction errors for the test set
err_test_total = calculate_reconstruction_error(pca_model, X_test_scaled)

# Separate errors for good and bad samples in the test set for analysis
err_test_good = err_test_total[y_test == 0]
err_test_bad = err_test_total[y_test == 1]

print("\nReconstruction Errors:")
print("Mean error for good test samples:", np.mean(err_test_good))
print("Mean error for bad test samples:", np.mean(err_test_bad))
print("Median error for good test samples:", np.median(err_test_good))
print("Median error for bad test samples:", np.median(err_test_bad))
print("Max error for good test samples:", np.max(err_test_good))
print("Max error for bad test samples:", np.max(err_test_bad))


# Determine optimal threshold using ROC curve
fpr, tpr, thresholds = metrics.roc_curve(y_test, err_test_total, pos_label=1)
optimal_idx = np.argmax(tpr - fpr)
loss_threshold = round_down_to_decimals(thresholds[optimal_idx], 13)
print(f"\nOptimal threshold: {loss_threshold}")

# Predict anomalies based on the threshold
Fulloutput = [1 if err > loss_threshold else 0 for err in err_test_total]

# Evaluate performance
cm = confusion_matrix(y_test, Fulloutput)
f1 = f1_score(y_test, Fulloutput)
auc = metrics.auc(fpr, tpr)

print("\nClassification Report:")
print(classification_report(y_test, Fulloutput))

print(f"F1 Score: {f1:.3f}")
print(f"AUC: {auc:.3f}")

# Plot ROC curve and Confusion Matrix
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, marker='.', label=f"AUC={auc:.3f}")
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right', prop={'size': 8.2})

plt.subplot(1, 2, 2)
ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap=plt.cm.Blues, ax=plt.gca())
plt.title('Confusion Matrix')

plt.tight_layout()
plt.show()

# Plot reconstruction errors
plt.figure(figsize=(10, 6))
colors = ['blue' if label == 0 else 'red' for label in y_test]
plt.bar(range(len(err_test_total)), err_test_total, color=colors)
plt.axhline(loss_threshold, color='black', linestyle='--', label=f'Threshold: {loss_threshold:.3f}')
plt.xlabel("Sample Index")
plt.ylabel("Reconstruction Error (L2 Distance)")
plt.title("Reconstruction Error on Test Set Samples")
plt.legend()
plt.show()